In [ ]:
%run ./1_config
import json, os, time, urllib.request
from pathlib import Path
from pyspark.sql import functions as F

class SilverLoader:
    def __init__(self):
        self.c = conf

    def distinct_ips(self):
        return [r.ip for r in spark.table(self.c.table_fqn(self.c.bronze_property_views)).select("ip").distinct().collect() if r.ip]

    def load_mock(self, ip):
        for p in [Path("data/fixtures/ipstack")/f"{ip}.json", Path(f"{self.c.fixture_base}/{ip}.json")]:
            if p.exists():
                return p.read_text(), 200
        return json.dumps({"ip": ip, "country_code": "AU", "country_name": "Australia", "region_name": "Victoria", "city": "Melbourne", "connection": {"isp": "Unknown"}, "security": {"is_proxy": False}}), 200

    def load_live(self, ip):
        key = os.getenv("IPSTACK_ACCESS_KEY", "")
        if not key:
            raise ValueError("IPSTACK_ACCESS_KEY required when MOCK_IPSTACK=false")
        url = f"http://api.ipstack.com/{ip}?access_key={key}&security=1"
        with urllib.request.urlopen(url, timeout=15) as resp:
            return resp.read().decode(), resp.getcode()

    def enrich_ips(self, ips):
        for ip in ips:
            try:
                body, status = self.load_mock(ip) if self.c.mock_ipstack else self.load_live(ip)
                if not self.c.mock_ipstack:
                    time.sleep(0.25)
                df = spark.createDataFrame([(ip, body, int(status))], "ip STRING, response_json STRING, http_status INT")
                df = df.withColumn("api_called_at", F.current_timestamp())
                df.write.format("delta").mode("append").saveAsTable(self.c.table_fqn(self.c.bronze_ipstack_raw))
            except Exception as e:
                err = spark.createDataFrame([(ip, str(e), 0)], "ip STRING, error_message STRING, http_status INT")
                err.withColumn("api_called_at", F.current_timestamp()).write.format("delta").mode("append").saveAsTable(self.c.table_fqn(self.c.bronze_ipstack_errors))

    def build_ip_dim(self):
        j = "ip STRING, country_code STRING, country_name STRING, region_name STRING, city STRING, latitude DOUBLE, longitude DOUBLE, time_zone STRUCT<id:STRING>, connection STRUCT<isp:STRING>, security STRUCT<is_proxy:BOOLEAN,proxy_type:STRING>"
        raw = spark.table(self.c.table_fqn(self.c.bronze_ipstack_raw))
        p = raw.withColumn("j", F.from_json("response_json", j))
        # Map AU region to state abbreviation for interstate analytics
        dim = p.select(
            "ip",
            F.col("j.country_code").alias("country_code"),
            F.col("j.country_name").alias("country_name"),
            F.col("j.region_name").alias("visitor_region"),
            F.col("j.city").alias("visitor_city"),
            F.col("j.latitude").alias("latitude"),
            F.col("j.longitude").alias("longitude"),
            F.col("j.time_zone.id").alias("timezone_id"),
            F.col("j.connection.isp").alias("isp"),
            F.when(F.col("j.security.proxy_type") == "vpn", True).otherwise(F.coalesce(F.col("j.security.is_proxy"), F.lit(False))).alias("is_vpn"),
            F.current_timestamp().alias("enriched_at"),
        ).dropDuplicates(["ip"])
        dim.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.silver_ip_dim))

    def build_listings(self):
        spark.table(self.c.table_fqn(self.c.bronze_listings)).drop("_ingested_at").write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.silver_listings))

    def build_visits(self):
        v = spark.table(self.c.table_fqn(self.c.bronze_property_views))
        l = spark.table(self.c.table_fqn(self.c.silver_listings))
        g = spark.table(self.c.table_fqn(self.c.silver_ip_dim))
        # Derive visitor_state from AU region names
        state_map = (
            F.when(F.col("visitor_region").contains("New South Wales"), "NSW")
            .when(F.col("visitor_region").contains("Victoria"), "VIC")
            .when(F.col("visitor_region").contains("Queensland"), "QLD")
            .when(F.col("visitor_region").contains("South Australia"), "SA")
            .when(F.col("visitor_region").contains("Western Australia"), "WA")
            .otherwise(F.lit("OTHER"))
        )
        device = F.when(F.col("user_agent").contains("iPhone"), F.lit("mobile")).otherwise(F.lit("desktop"))
        out = (v.join(l, "property_id", "inner")
               .join(g, "ip", "left")
               .withColumn("visitor_state", state_map)
               .withColumn("device_type", device)
               .withColumnRenamed("state", "listing_state")
               .select("event_id","session_id","event_ts","user_id","ip","property_id",
                       "view_duration_sec","enquiry_flag","favorite_flag","event_date",
                       F.col("country_name").alias("visitor_country"),
                       "visitor_region","visitor_city","visitor_state","timezone_id","device_type",
                       "suburb","listing_state","property_type","price_range","is_luxury","price"))
        out.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(self.c.table_fqn(self.c.silver_visits_enriched))

    def run(self):
        ips = self.distinct_ips()
        print(f"Enriching {len(ips)} visitor IPs (mock={self.c.mock_ipstack})")
        self.enrich_ips(ips)
        self.build_ip_dim()
        self.build_listings()
        self.build_visits()
        print("✅ Silver complete")

SilverLoader().run()

